# CRUD Operations in ChromaDB

**Level:** Scratch → Intermediate
**Runs on:** Google Colab (Free CPU)

**CRUD** = **C**reate, **R**ead, **U**pdate, **D**elete — the four core actions used by almost every database.

In this notebook, you will learn how to use ChromaDB to:
1. create a collection
2. add documents with ids and metadata
3. read documents by id and by semantic search
4. update and upsert existing items
5. delete entries by id or filter
6. check the collection state with `count()`, `peek()`, and `get()`


### 1. Install ChromaDB
Run this once in your environment.

In [ ]:
!pip install -q chromadb

### 2. Create a Client and Collection

An in-memory client is simple for learning because it resets when the runtime restarts.

In [ ]:
import chromadb

client = chromadb.Client()
collection = client.create_collection(name="crud_demo")

print("Collection created:", collection.name)
print("Current document count:", collection.count())

### 3. CREATE — Add Documents

Each document should have a unique id. `documents` hold the actual text, and `metadatas` help organize records.

In [ ]:
collection.add(
    documents=[
        "Python is a popular programming language for AI.",
        "The weather today is sunny and warm.",
        "Machine learning models learn patterns from data.",
        "I enjoy playing football on weekends.",
        "Deep learning uses neural networks with many layers."
    ],
    metadatas=[
        {"category": "tech"},
        {"category": "weather"},
        {"category": "tech"},
        {"category": "sports"},
        {"category": "tech"}
    ],
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"]
)

print("Documents added! Total count:", collection.count())

**Important:** If you try to add a document with an existing id, ChromaDB raises an error. Use `upsert()` if you want add-or-update behavior.

### 4. READ — Preview the Collection

`peek()` gives a quick sample without needing a similarity query.

In [ ]:
preview = collection.peek(limit=3)

for doc_id, doc in zip(preview["ids"], preview["documents"]):
    print(doc_id, "->", doc)

### 5. READ — Fetch Documents by ID

Use `get()` when you already know the exact id or ids you want. This is not a semantic search; it is a direct lookup.

In [ ]:
result = collection.get(ids=["doc1", "doc3"])

for doc_id, doc, meta in zip(result["ids"], result["documents"], result["metadatas"]):
    print(f"{doc_id} | {meta} | {doc}")

### 6. READ — Semantic Query (`query`)

This is the key vector-search feature in ChromaDB: you ask for meaning, not exact text matches.

In [ ]:
results = collection.query(
    query_texts=["Tell me about AI and neural networks"],
    n_results=2
)

for doc_id, doc, distance in zip(results["ids"][0], results["documents"][0], results["distances"][0]):
    print(f"{doc_id} | distance={distance:.4f} | {doc}")

### 7. READ — Filtered Query with `where`

You can combine semantic search with exact metadata filters. Here, we only search within the `sports` category.

In [ ]:
results = collection.query(
    query_texts=["something fun to do outside"],
    n_results=3,
    where={"category": "sports"}
)

for doc_id, doc, distance in zip(results["ids"][0], results["documents"][0], results["distances"][0]):
    print(f"{doc_id} | distance={distance:.4f} | {doc}")

### 8. UPDATE — Replace Existing Documents

Use `update()` when you want to change the text and/or metadata of an item that already exists.

In [ ]:
collection.update(
    ids=["doc4"],
    documents=["I enjoy playing football and basketball on weekends."],
    metadatas=[{"category": "sports", "updated": True}]
)

updated = collection.get(ids=["doc4"])
print(updated["documents"])
print(updated["metadatas"])

### 9. UPSERT — Add or Update in One Step

`upsert()` is helpful when you are not sure whether the id already exists. If it exists, it updates it; if not, it creates it.

In [ ]:
collection.upsert(
    documents=[
        "Deep learning uses neural networks with many layers and GPUs.",
        "The Eiffel Tower is located in Paris."
    ],
    ids=["doc5", "doc6"]
)

print("Total count after upsert:", collection.count())
print(collection.get(ids=["doc5", "doc6"])["documents"])

### 10. DELETE — Remove Documents by ID

You can delete one or many records by passing their ids.

In [ ]:
collection.delete(ids=["doc2"])

print("Total count after delete:", collection.count())
print("Remaining ids:", collection.get()["ids"])

### 11. DELETE — Remove Documents by Metadata Filter

This is useful when you want to delete all records that match a condition, such as `category == "tech"`.

In [ ]:
collection.delete(where={"category": "tech"})

print("Total count after category delete:", collection.count())
print("Remaining ids:", collection.get()["ids"])

### 12. Final State Check

This helps confirm what remains in the collection after all operations.

In [ ]:
final_state = collection.get()

print("Final document count:", collection.count())
for doc_id, doc, meta in zip(final_state["ids"], final_state["documents"], final_state["metadatas"]):
    print(f"{doc_id} | {meta} | {doc}")

### 13. Key Takeaways

| Operation | Method | Notes |
|---|---|---|
| Create collection | `client.create_collection()` | Creates a new collection |
| Create/Add | `collection.add()` | Inserts new documents; duplicate ids raise an error |
| Read (exact) | `collection.get()` | Fetches by id, no similarity involved |
| Read (semantic) | `collection.query()` | Finds nearest matches based on meaning |
| Read (preview) | `collection.peek()` | Shows a quick sample of the collection |
| Update | `collection.update()` | Modifies an existing document by id |
| Add or Update | `collection.upsert()` | Creates if missing, updates if it exists |
| Delete | `collection.delete()` | Removes by id list or metadata filter |

### 14. Practice Exercises
1. Add 5 new documents, then use `upsert()` to update 2 of them and create 1 brand-new record in one call.
2. Delete all documents whose `category` matches a value you choose and confirm the result with `collection.count()`.
3. Write a helper function called `safe_add(collection, doc_id, text)` that uses `upsert()` so duplicate ids do not cause errors.